# 🦾 FingerPush: designing a reinforcement-learning environment

In this tutorial you teach a fixed-base **3-DOF finger** to **push a cube to a goal** — first to one fixed spot, then to *any* goal you give it. You do this by **designing the environment**, not by touching the learning algorithm.

The heavy machinery is given to you and marked **⚙️ PROVIDED**:
- **Physics** — thousands of copies of the world stepped in parallel on the GPU with [**MuJoCo Warp**](https://github.com/google-deepmind/mujoco_warp).
- **The RL algorithm** — a standard PPO trainer.
- Observation/reward normalisation, logging, checkpointing and rendering.

What you implement — marked **🧩 YOUR CODE** — is the part that actually defines the task and decides whether the robot can learn it at all:

| Concept | Method(s) | The question it answers |
|---|---|---|
| **Observation** | `_compute_obs` | What is the policy allowed to see? |
| **Action / control** | `_process_action`, `_pd_torque` | How does an action become motion? |
| **Reward** | `_compute_reward` | What does "good" mean? |
| **Episode** | `_compute_dones`, `_update_goal_timer` | When does an attempt end — and did it succeed? |
| **Goals** (Part B) | `_resample_goals` | Where is the cube supposed to go? |

These are *the* levers of environment design, and getting them right is most of the work in robot learning with reinforcement learning.

> ⚙️ **Before you run anything:** this notebook needs a **GPU**. On Colab: **Runtime → Change runtime type → Hardware accelerator → GPU** (a T4 is plenty).

## How to use this notebook

1. Run the setup sections (**1–3**) top to bottom — all ⚙️ PROVIDED.
2. In **Section 4** you complete the environment: fill in every 🧩 `YOUR CODE` method. Each one raises `NotImplementedError` until you do — **Section 5** then validates your work.
3. **Part A** (Section 8): train the finger to push the cube to **one fixed goal**.
4. **Part B** (Section 9): with a **one-method** change, make the goal **random every episode**, so a *single* policy learns to push the cube anywhere in a region.

Two markers appear throughout:

- **⚙️ PROVIDED** — read it if you're curious, but you don't need to edit it.
- **🧩 YOUR CODE** — your job. Each method's docstring tells you exactly what to compute (formula included).

## 1&nbsp;· Install dependencies

Colab already ships PyTorch with CUDA. We add MuJoCo, MuJoCo-Warp and Warp.

In [ ]:
# MuJoCo Warp is still young — pin nothing, take the latest wheels.
# (tensorboard ships with Colab already; listed for completeness.)
%pip install -q mujoco mujoco-warp warp-lang mediapy tensorboard robot_descriptions torch

## 2&nbsp;· Get the robot model

The FingerEdu meshes come from the
[`robot_descriptions`](https://github.com/robot-descriptions/robot_descriptions.py)
package (which caches them from `example-robot-data`). That package only ships the
**URDF** — no actuators, no scene — so the two small MJCF files from the course repo
(`finger_edu.xml` with the motors, and the scene with the cube + target marker) are
inlined below as strings and written next to a `meshdir` pointing into the package
cache. The resulting model is identical to the course repo's
`finger_edu_scene_cube.xml`.

In [ ]:
import os, pathlib

# Downloads (once) and caches the FingerEdu description; we only use its meshes.
from robot_descriptions import finger_edu_description

MESH_DIR = os.path.join(finger_edu_description.PACKAGE_PATH, "meshes")

# MJCF robot (from the course repo's finger_edu.xml) — the URDF in the package has
# no actuators, so the <motor> definitions here are what make the finger controllable.
FINGER_EDU_XML = f"""
<mujoco model="fingeredu">
  <compiler angle="radian" meshdir="{MESH_DIR}"/>

  <asset>
    <mesh name="base_back" content_type="model/stl" file="base_back.stl"/>
    <mesh name="base_front" content_type="model/stl" file="base_front.stl"/>
    <mesh name="base_side_left" content_type="model/stl" file="base_side_left.stl"/>
    <mesh name="base_top" content_type="model/stl" file="base_top.stl"/>
    <mesh name="upper_link" content_type="model/stl" file="upper_link.stl"/>
    <mesh name="middle_link" content_type="model/stl" file="middle_link.stl"/>
    <mesh name="lower_link" content_type="model/stl" file="lower_link.stl"/>
  </asset>

  <worldbody>
    <body name="finger_base_link" pos="0 0 0.05">
      <geom pos="-0.17995 0 0.283" type="mesh" contype="0" conaffinity="0" rgba="0.6 0.6 0.6 1" mesh="base_back"/>
      <geom pos="0.0255 0 0.283" type="mesh" contype="0" conaffinity="0" rgba="0.6 0.6 0.6 1" mesh="base_front"/>
      <geom pos="0.0255 0.02 0.363" quat="1 0 0 0" contype="0" conaffinity="0" type="mesh" rgba="0.6 0.6 0.6 1" mesh="base_side_left"/>
      <geom pos="0.0255 0 0.363" quat="1 0 0 0" contype="0" conaffinity="0" type="mesh" rgba="0.6 0.6 0.6 1" mesh="base_top"/>
      <body name="finger_upper_link" pos="0 0 0.283">
        <inertial pos="-0.079 0 0" quat="0.531109 0.531109 0.466822 0.466822" mass="0.14854" diaginertia="0.000416469 0.00041 2.35312e-05"/>
        <joint name="finger_base_to_upper_joint" pos="0 0 0" axis="-1 0 0" range="-1.5708 1.5708" actuatorfrcrange="-1 1"/>
        <geom pos="0.0195 0 0" quat="1 0 0 0" type="mesh" contype="0" conaffinity="0" rgba="0.6 0.6 0.6 1" mesh="upper_link"/>
        <body name="finger_middle_link" pos="0 -0.014 0">
          <inertial pos="0 -0.019 -0.079" quat="0.705644 0.0454575 -0.0454575 0.705644" mass="0.14854" diaginertia="0.000416469 0.00041 2.35312e-05"/>
          <joint name="finger_upper_to_middle_joint" pos="0 0 0" axis="0 1 0" range="-1.5708 1.5708" actuatorfrcrange="-1 1"/>
          <geom type="mesh" rgba="0.6 0.6 0.6 1" contype="0" conaffinity="0" mesh="middle_link"/>
          <body name="finger_lower_link" pos="0 -0.03745 -0.16">
            <inertial pos="0 -0.0087543 -0.106445" quat="0.999999 0.00170537 0 0" mass="0.0407" diaginertia="0.000158198 0.000158193 1.17238e-06"/>
            <joint name="finger_middle_to_lower_joint" pos="0 0 0" axis="0 1 0" range="-3.14159 3.14159" actuatorfrcrange="-1 1"/>
            <geom type="mesh" rgba="0.6 0.6 0.6 1" contype="1" conaffinity="0" mesh="lower_link"/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="finger_base_to_upper_joint" joint="finger_base_to_upper_joint"/>
    <motor name="finger_upper_to_middle_joint" joint="finger_upper_to_middle_joint"/>
    <motor name="finger_middle_to_lower_joint" joint="finger_middle_to_lower_joint"/>
  </actuator>
</mujoco>
"""

# Scene (from the course repo's finger_edu_scene_cube.xml): floor, cube, target marker.
# The marker matches the env's actual target, target_xy = (0.3, -0.05).
SCENE_XML = """
<mujoco model="finger edu scene">
  <option timestep="0.005"/>
  <include file="finger_edu.xml"/>

  <statistic center="0 0 0.1" extent="0.8"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="-130" elevation="-20"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3"
      markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="box" pos="0.1 -0.05 0.05">
      <freejoint/>
      <geom name="box_geom" type="box" mass="0.01" size="0.05 0.05 0.05" rgba="0.8 0.2 0.2 1"/>
    </body>
    <geom name="target_marker" type="sphere" size="0.05" pos="0.3 -0.05 0.05" rgba="0 1 0 1" contype="0" conaffinity="0"/>
  </worldbody>
</mujoco>
"""

MODEL_DIR = pathlib.Path("finger_edu_model")
MODEL_DIR.mkdir(exist_ok=True)
(MODEL_DIR / "finger_edu.xml").write_text(FINGER_EDU_XML)
(MODEL_DIR / "finger_edu_scene_cube.xml").write_text(SCENE_XML)

XML_PATH = os.path.abspath(MODEL_DIR / "finger_edu_scene_cube.xml")
assert os.path.exists(XML_PATH), XML_PATH
print("meshes:", MESH_DIR)
print("model :", XML_PATH)

### 2b&nbsp;· Sanity check: zero-torque rollout

Before touching Warp or training, simulate a few seconds of **passive dynamics**
(`ctrl = 0`) with plain CPU MuJoCo and record a small GIF. If the model and meshes
loaded correctly you'll see the finger swing down under gravity next to the red cube
and the green target marker. The GIF is also saved to `zero_torque_preview.gif`.

In [ ]:
# Passive rollout: zero torque on all three motors, gravity does the rest.
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # off-screen rendering
import mujoco
import mediapy as media

mjm = mujoco.MjModel.from_xml_path(XML_PATH)
mjd = mujoco.MjData(mjm)

FPS = 25
DURATION = 3.0   # seconds of simulated time
steps_per_frame = max(1, round(1.0 / (FPS * mjm.opt.timestep)))

frames = []
renderer = mujoco.Renderer(mjm, height=240, width=320)
while mjd.time < DURATION:
    mjd.ctrl[:] = 0.0                       # zero torque
    for _ in range(steps_per_frame):
        mujoco.mj_step(mjm, mjd)
    renderer.update_scene(mjd, camera=-1)
    frames.append(renderer.render())
renderer.close()

GIF_PATH = "zero_torque_preview.gif"
media.write_video(GIF_PATH, frames, fps=FPS, codec="gif")
print(f"saved {GIF_PATH} ({len(frames)} frames, {DURATION:.1f}s sim time)")
media.show_video(frames, fps=FPS, codec="gif")

## 3&nbsp;· Imports & GPU check

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # off-screen rendering later

import time, numpy as np, torch, torch.nn as nn
import mujoco, warp as wp
import mujoco_warp as mjw
wp.config.quiet = True

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU"
DEVICE = "cuda"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("warp", wp.__version__)

## 4 · Designing the environment

An RL environment is a loop. One call to `env.step(action)` runs this pipeline on **every world at once** — the methods you write are marked 🧩:

```
   observation ──▶ [ PPO policy ] ──▶ action  (in [-1, 1])
        ▲                                │
        │                        🧩 _process_action     action → joint-position target
        │                                │
        │                        🧩 _pd_torque   ├─ repeated `decimation` times:
        │                                │        │   target → torque → one physics substep
        │                                │        └─ 🧩 _update_goal_timer  (held on goal?)
        │                                ▼
   🧩 _compute_obs ◀── reward, done ◀── 🧩 _compute_reward + 🧩 _compute_dones
```

Five design decisions define the whole task:

**1 · Observation — `_compute_obs`.** The policy is a function of the observation *only*. If something the robot needs isn't in here, the task is **partially observed** and may be unlearnable. Key choice: we observe the cube's position **relative to the goal** — which is exactly what lets one policy chase *different* goals in Part B.

**2 · Action & control — `_process_action`, `_pd_torque`.** The policy outputs numbers in `[-1, 1]`; the robot has motors. We read the action as a **joint-position target** and use a **PD controller** to convert it to torque. "Position target in, torque out" is a standard, robust action space for robot learning: the policy says *where* to move, the controller decides *how hard* to push. Note the physics runs `decimation` substeps per policy step, so control is smooth and the policy runs at a sensible rate.

**3 · Reward — `_compute_reward`.** The hardest part. A pure reward-on-success signal is almost never hit by random exploration, so we **shape** the reward with dense terms that continuously point toward the goal. You get a working starter, then improve it.

**4 · Episode boundaries — `_compute_dones`, `_update_goal_timer`.** When does an attempt end? Separate a real ending (**terminated** — success or failure) from simply running out of time (**truncated**); the trainer treats them differently. And success must be *held* on the goal, not a lucky fly-through — that is what the goal timer enforces.

**5 · Goals — `_resample_goals`.** Where must the cube go? One fixed spot in Part A; a fresh random one each episode in Part B.

### The provided GPU plumbing  (⚙️, skim once)
The model is loaded on the CPU with MuJoCo, then handed to Warp: `mjw.put_data(..., nworld=num_envs)` stacks `num_envs` copies of the world (every state array gains a leading `num_envs` dimension), and `wp.to_torch(d.qpos)` returns a torch tensor that **shares GPU memory** with the simulator — reading or writing it reads or writes the physics state directly, no copies. One physics substep is captured into a **CUDA graph** and replayed each step for speed, and worlds that finish an episode are **auto-reset in place** so the rollout never stalls. You never call any of this directly; you just write batched torch over `(num_envs, …)` tensors.

### 🧩 Your task — complete `FingerPushVecEnv`

Fill in the six 🧩 methods in the cell below (each docstring gives the formula). A good order:

1. **`_compute_obs`** — assemble the observation, and set `self.num_obs` to match its width.
2. **`_process_action`** + **`_pd_torque`** — action → joint-position target → clamped torque.
3. **`_update_goal_timer`** — count substeps the cube stays inside the goal radius.
4. **`_compute_reward`** — a **working starter is already given**; run it first, improve it later.
5. **`_compute_dones`** — terminated (success / failure) vs truncated (time-out).

Nothing trains until these are done. When you finish, run **Section 5** to check shapes and stepping.

In [ ]:
import numpy as np
import mujoco
import warp as wp
import torch
import mujoco_warp as mjw


class FingerPushVecEnv:
    """Massively-parallel MuJoCo-Warp FingerPush environment — TUTORIAL SCAFFOLD.

    `num_envs` copies of the finger+cube world are stepped together on the GPU.
    The GPU plumbing is PROVIDED (marked  ⚙️ PROVIDED) and you should not need to
    touch it: the Warp model/data, CUDA-graph stepping, zero-copy torch views,
    auto-reset, and the terminated-vs-truncated bookkeeping the trainer relies on.

    The environment-DESIGN decisions are YOURS (marked  🧩 YOUR CODE). Each one
    raises NotImplementedError until you fill it in:

        _compute_obs        what the policy is allowed to see
        _process_action     raw policy action  -> joint-position target
        _pd_torque          joint-position target -> motor torque (PD control law)
        _update_goal_timer  how long the cube has been parked on the goal
        _compute_reward     the learning signal  (a WORKING STARTER is given)
        _compute_dones      when, and WHY, an episode ends (terminate vs truncate)

    Part B adds one more, in a subclass:  _resample_goals.
    """

    def __init__(self, xml_path, num_envs=2048, device="cuda", decimation=4,
                 episode_length_s=2.0, randomize_init=True, seed=0):
        self.num_envs = num_envs
        self.device = torch.device(device)
        self.decimation = decimation

        # ⚙️ PROVIDED: load the model on CPU with plain MuJoCo, then hand it to Warp.
        self.mjm = mujoco.MjModel.from_xml_path(xml_path)
        self.mjd = mujoco.MjData(self.mjm)
        mujoco.mj_forward(self.mjm, self.mjd)

        self.dt = self.mjm.opt.timestep
        self.control_dt = self.dt * self.decimation
        # episode_step counts POLICY steps (one per control_dt = decimation substeps)
        self.steps_per_episode = int(episode_length_s / self.control_dt)

        self.m = mjw.put_model(self.mjm)
        # njmax is the per-world constraint-row budget; the auto value (~5) is far too
        # tight for this contact-rich scene and silently drops constraints. 128 is safe.
        self.d = mjw.put_data(self.mjm, self.mjd, nworld=num_envs, njmax=128)

        # ⚙️ PROVIDED: zero-copy torch views onto the batched Warp state (num_envs, ...).
        # Reading/writing these tensors reads/writes the simulator directly — no copies.
        self.qpos = wp.to_torch(self.d.qpos)     # (num_envs, nq=10): 3 joints + cube(pos3,quat4)
        self.qvel = wp.to_torch(self.d.qvel)     # (num_envs, nv=9):  3 joints + cube(lin3,ang3)
        self.ctrl = wp.to_torch(self.d.ctrl)     # (num_envs, nu=3):  motor torques
        self.xpos = wp.to_torch(self.d.xpos)     # (num_envs, nbody, 3): body world positions
        self.qacc_ws = wp.to_torch(self.d.qacc_warmstart)
        self.fingertip_body_id = mujoco.mj_name2id(
            self.mjm, mujoco.mjtObj.mjOBJ_BODY, "finger_lower_link")

        # -- task / controller parameters --------------------------------------
        self.num_dof = 3
        self.kp = 2.5                 # PD position gain
        self.kd = 0.2                 # PD velocity gain
        self.action_scale = 0.5       # policy action in [-1,1] -> +-0.5 rad around default
        self.torque_limit = 1.0       # actuatorfrcrange in the XML is [-1, 1]
        self.default_joint_pos = torch.tensor([0.0, 0.5, -0.75], device=self.device)

        self.reach_goal_threshold = 50   # the cube must stay inside the goal radius for...
        self.goal_radius = 0.05          # ...this many sim substeps (0.05 m) to count as held

        # initial state captured from the loaded model
        self.qpos_init = torch.tensor(self.mjd.qpos, device=self.device, dtype=torch.float32)
        self.qvel_init = torch.tensor(self.mjd.qvel, device=self.device, dtype=torch.float32)
        self.cube_init_xy = self.qpos_init[3:5].clone()
        self.randomize_init = randomize_init

        # per-world goal + per-world start-distance (both filled by _init_world_state).
        # target_xy is (num_envs, 2) so every world CAN have its own goal — Part B uses
        # that; Part A just gives every world the same one.
        self.target_xy = torch.zeros(num_envs, 2, device=self.device)
        self.d_init = torch.ones(num_envs, device=self.device)

        # -- spaces ------------------------------------------------------------
        self.num_actions = self.num_dof
        # 🧩 YOUR CODE: set num_obs so it matches what _compute_obs returns.
        #   Suggested layout (24): joint_pos(3) joint_vel(3) cube_pos_rel(2) cube_quat(4)
        #                          cube_lin_vel(3) cube_ang_vel(3) tip_to_cube(3) prev_action(3)
        self.num_obs = None          # <-- replace with the right integer

        # -- episode bookkeeping (all on GPU) ----------------------------------
        self.gen = torch.Generator(device=self.device).manual_seed(seed)
        self.episode_step = torch.zeros(num_envs, dtype=torch.long, device=self.device)
        self.reach_goal_timer = torch.zeros(num_envs, dtype=torch.long, device=self.device)
        self.prev_action = torch.zeros(num_envs, self.num_actions, device=self.device)
        self.action = torch.zeros(num_envs, self.num_actions, device=self.device)
        self.ep_return = torch.zeros(num_envs, device=self.device)
        self.ep_len = torch.zeros(num_envs, dtype=torch.long, device=self.device)

        self._step_graph = None
        self._forward_graph = None
        self._init_world_state(torch.arange(num_envs, device=self.device))
        self._build_graphs()

    # ========================= ⚙️ PROVIDED plumbing ==========================
    def _build_graphs(self):
        """Capture one physics substep and one forward pass into CUDA graphs, so each
        can later be replayed with a single launch (this is what makes Warp fast)."""
        self.ctrl.zero_()
        mjw.step(self.m, self.d)       # warmup / kernel compile
        mjw.forward(self.m, self.d)
        wp.synchronize()
        with wp.ScopedCapture() as cap:
            mjw.step(self.m, self.d)
        self._step_graph = cap.graph
        with wp.ScopedCapture() as cap:
            mjw.forward(self.m, self.d)
        self._forward_graph = cap.graph

    def _physics_substep(self):
        wp.capture_launch(self._step_graph)

    def _refresh_derived(self):
        """Recompute derived state (xpos, contacts, ...) from qpos/qvel. Needed after an
        in-place reset writes qpos, so the first observation of an episode isn't stale."""
        wp.capture_launch(self._forward_graph)

    def _init_world_state(self, ids):
        """⚙️ PROVIDED: reset the given world indices to a (randomized) initial state,
        (re)draw their goal, and recompute their per-world start-distance d_init."""
        n = ids.shape[0]
        self.qpos[ids] = self.qpos_init
        self.qvel[ids] = self.qvel_init
        self.qacc_ws[ids] = 0.0        # clear solver warmstart so a bad world can't re-seed NaNs
        if self.randomize_init:
            self.qpos[ids, 3:5] += 0.02 * (2 * torch.rand(n, 2, generator=self.gen, device=self.device) - 1)
            self.qpos[ids, 0:3] += 0.05 * (2 * torch.rand(n, 3, generator=self.gen, device=self.device) - 1)
        # draw each world's goal (fixed in Part A; random in the Part B subclass)...
        self._resample_goals(ids)
        # ...and normalise the distance reward per world so "distance at reset" ~= 1.0.
        # (In Part B the goal differs per world, so d_init must be per world too.)
        self.d_init[ids] = torch.linalg.norm(
            self.cube_init_xy - self.target_xy[ids], dim=-1).clamp_min(1e-4)
        self.episode_step[ids] = 0
        self.reach_goal_timer[ids] = 0
        self.prev_action[ids] = 0.0
        self.action[ids] = 0.0

    def _resample_goals(self, ids):
        """⚙️ PROVIDED (Part A): every world shares one fixed goal.
        Part B OVERRIDES this method to draw a random goal per world."""
        self.target_xy[ids] = torch.tensor([0.30, -0.05], device=self.device)

    # ============================ 🧩 YOUR CODE ===============================
    def _compute_obs(self):
        """🧩 YOUR CODE: build the observation the policy sees each step.

        You are given the raw simulator state below (all batched, shape (num_envs, ...)).
        Two DESIGN decisions matter most:

          (1) Encode the cube position RELATIVE TO THE GOAL, not in absolute
              coordinates:   cube_pos_rel = cube_pos[:, 0:2] - self.target_xy   -> (N, 2)
              This is the ONLY place the goal enters the observation. It is exactly
              what lets a single network pursue DIFFERENT goals in Part B: move the
              goal, and this number changes, so the policy can respond. Observe the
              cube absolutely instead and the task becomes partially observed — the
              policy cannot see where it is being asked to push.

          (2) Give the policy the fingertip->cube vector so it can learn to make and
              keep contact:  tip_to_cube = cube_pos - tip_pos                   -> (N, 3)

        Return torch.cat([...], dim=-1) with shape (num_envs, num_obs). Suggested order
        (24 dims): joint_pos(3) joint_vel(3) cube_pos_rel(2) cube_quat(4)
                   cube_lin_vel(3) cube_ang_vel(3) tip_to_cube(3) prev_action(3).
        Whatever order you pick, set self.num_obs to match (and mirror it in the
        rollout-video helper, which rebuilds the obs by hand for CPU rendering).
        """
        joint_pos = self.qpos[:, 0:3]
        joint_vel = self.qvel[:, 0:3]
        cube_pos = self.qpos[:, 3:6]
        cube_quat = self.qpos[:, 6:10]
        cube_lin_vel = self.qvel[:, 3:6]
        cube_ang_vel = self.qvel[:, 6:9]
        tip_pos = self.xpos[:, self.fingertip_body_id, :]
        # TODO(1): cube position relative to the goal  ->  cube_pos_rel = ...
        # TODO(2): fingertip -> cube vector            ->  tip_to_cube  = ...
        # TODO(3): concatenate everything (see the order in the docstring above)
        # You can also underdefine the system using less obs to see how it affects
        # learning and to understand which are the most important observations
        raise NotImplementedError("implement _compute_obs (then delete this line)")

    def _process_action(self, action):
        """🧩 YOUR CODE: map the policy's raw action in [-1, 1] to a joint-position
        target. Scale it (self.action_scale) so one step is a small nudge, and centre
        it on a sensible resting posture (self.default_joint_pos):

            joint_pos_target = action_scale * action + default_joint_pos

        Returns a (num_envs, 3) tensor of target joint angles."""
        raise NotImplementedError("implement _process_action (then delete this line)")

    def _pd_torque(self, joint_pos_target, joint_pos, joint_vel):
        """🧩 YOUR CODE: the PD control law. Turn a joint-position target into a motor
        torque and clamp it to the actuator limit. This is the low-level controller the
        RL policy drives — "position targets in, torques out" is a very common and
        robust action-space design for robot learning.

            tau = kp * (joint_pos_target - joint_pos) - kd * joint_vel
            tau = clamp(tau, -torque_limit, +torque_limit)
        """
        raise NotImplementedError("implement _pd_torque (then delete this line)")

    def _update_goal_timer(self):
        """🧩 YOUR CODE: called once per physics substep. Success should not be a
        single lucky frame — the cube must STAY on the goal. Count up for worlds whose
        cube is within self.goal_radius of self.target_xy, and reset the count to 0 for
        worlds that are not:

            cube_xy = self.qpos[:, 3:5]
            dist    = ||cube_xy - self.target_xy||          (per world)
            inside  = dist < self.goal_radius
            self.reach_goal_timer = where(inside, self.reach_goal_timer + 1, 0)
        """
        raise NotImplementedError("implement _update_goal_timer (then delete this line)")

    def _compute_reward(self):
        """🧩 YOUR CODE (a working STARTER is provided): return (reward, terms) where
        reward is a (num_envs,) tensor and terms is a dict of scalar means for logging.

        The starter below ALREADY TRAINS: a dense term that grows as the cube nears the
        goal, plus an 'approach' term that pulls the fingertip toward the cube so the
        policy discovers contact (without it the reward is flat until a lucky touch and
        there is nothing to learn from). It will drive the cube toward the goal — but it
        tends to OVERSHOOT and never settle, because nothing rewards arriving slowly or
        holding still.

        IMPROVE IT (see the markdown above this cell) by adding more or removing terms such as
        for improving precision, reduce aggresive behaviour etc
          * precision  :  torch.exp(-distance / 0.02)                sharp gradient in the last cm
          * brake      : -cube_speed * torch.exp(-distance / 0.1)    punish speed near the goal
          * hold bonus : +50 * (self.reach_goal_timer > self.reach_goal_threshold)   sparse success
          * action_rate: -||self.action - self.prev_action||         encourage smoother motions
        Add each as its own entry in `terms` so you can watch it separately in TensorBoard.
        """
        dt = self.control_dt
        cube_xy = self.qpos[:, 3:5]
        distance = torch.linalg.norm(cube_xy - self.target_xy, dim=-1)

        # dense distance reward: ~1.0 at the start, -> 0 as the cube reaches the goal
        linear_distance_reward = torch.clamp(1.0 - distance / self.d_init, 0.0, 1.0)

        # approach shaping: reward the fingertip for getting near the cube. Use the
        # HORIZONTAL (xy) distance — the lower link hangs ~0.16 m below its body frame,
        # so a full 3D distance would reward pressing straight down instead of lining
        # up behind the cube.
        tip_pos = self.xpos[:, self.fingertip_body_id, :]
        tip_to_cube = torch.linalg.norm(cube_xy - tip_pos[:, 0:2], dim=-1)
        approach_reward = 1.0 - torch.tanh(tip_to_cube / 0.1)

        r_distance = 2.0 * linear_distance_reward * dt
        r_approach = 0.5 * approach_reward * dt
        reward = r_distance + r_approach
        terms = {"distance": r_distance.mean().item(),
                 "approach": r_approach.mean().item()}
        # TODO(improve): add precision / brake / hold-bonus / action_rate terms, then
        #                fold them into `reward` and record each in `terms`.
        return reward, terms

    def _compute_dones(self):
        """🧩 YOUR CODE: decide which episodes end this step, and WHY. The trainer
        treats the two reasons differently, so you must return them separately:

          terminated : the episode reached a real end state --
                         * success : cube held on the goal
                                     (self.reach_goal_timer > self.reach_goal_threshold + 1)
                         * failure : cube knocked out of reach
                                     (distance > self.d_init + 0.1)
          truncated  : nothing ended it except the TIME LIMIT, i.e.
                         (self.episode_step >= self.steps_per_episode) AND NOT terminated

        Why separate them? A time-out is not a real terminal state — the task did not
        conclude, we just stopped looking. The PPO loop bootstraps the value of the
        final state on truncation, instead of wrongly treating a time-out as a
        zero-value death. (A numerical blow-up safety-termination is OR'd in for you
        inside step(), so you don't have to handle NaNs here.)

        Return (terminated, truncated, goal_reached) as boolean (num_envs,) tensors.
        """
        raise NotImplementedError("implement _compute_dones (then delete this line)")

    # ============================ ⚙️ PROVIDED API ============================
    @torch.no_grad()
    def reset(self):
        """⚙️ PROVIDED. Reset every world, stagger the episode clocks so the batch does
        not time out in lockstep, and return the first (batched) observation."""
        self._init_world_state(torch.arange(self.num_envs, device=self.device))
        if self.randomize_init:
            self.episode_step = torch.randint(0, self.steps_per_episode, (self.num_envs,),
                                              generator=self.gen, device=self.device)
        self.ep_return.zero_(); self.ep_len.zero_()
        self._refresh_derived()
        return self._compute_obs()

    @torch.no_grad()
    def step(self, action):
        """⚙️ PROVIDED. Drives YOUR methods (marked below), then handles the physics
        substeps, NaN safety, auto-reset, truncation bootstrap data and logging. You
        should not need to edit this."""
        action = torch.clamp(action.to(self.device), -1.0, 1.0)
        self.action = action.clone()

        joint_pos_target = self._process_action(action)                 # 🧩 your code

        for _ in range(self.decimation):
            joint_pos = self.qpos[:, 0:3]
            joint_vel = self.qvel[:, 0:3]
            torque = self._pd_torque(joint_pos_target, joint_pos, joint_vel)   # 🧩 your code
            self.ctrl[:, 0:3] = torque
            self._physics_substep()
            self._update_goal_timer()                                   # 🧩 your code

        self.episode_step += 1
        reward, terms = self._compute_reward()                          # 🧩 your code
        terminated, truncated, goal_reached = self._compute_dones()     # 🧩 your code

        # safety: with a 10 g box + hard contact, a few worlds among thousands can go
        # numerically unstable. Zero their reward and force a reset so a NaN never
        # reaches the observation or the policy (standard for large-batch GPU sim).
        bad = ~torch.isfinite(self.qpos).all(dim=1) | ~torch.isfinite(self.qvel).all(dim=1)
        if bad.any():
            reward = torch.where(bad, torch.zeros_like(reward), reward)
            terminated = terminated | bad
            truncated = truncated & ~terminated
        done = terminated | truncated

        self.ep_return += reward
        self.ep_len += 1
        info = {"reward_terms": terms, "truncated": truncated, "goal_reached": goal_reached}
        if done.any():
            info["episode_return"] = self.ep_return[done].mean().item()
            info["episode_length"] = self.ep_len[done].float().mean().item()
            info["num_done"] = int(done.sum().item())
            info["goal_reached_frac"] = float(goal_reached[done].float().mean().item())
        self.prev_action = action.clone()

        # observation of the state each episode actually ended in (pre-reset); the
        # trainer bootstraps V(final_obs) for truncated episodes.
        final_obs = torch.nan_to_num(self._compute_obs())
        info["final_obs"] = final_obs

        # auto-reset finished worlds so the rollout never stalls, then refresh their obs
        reset_ids = done.nonzero(as_tuple=False).squeeze(-1)
        if reset_ids.numel() > 0:
            self._init_world_state(reset_ids)
            self.ep_return[reset_ids] = 0.0
            self.ep_len[reset_ids] = 0
            self._refresh_derived()
            obs = torch.nan_to_num(self._compute_obs())
        else:
            obs = final_obs
        return obs, reward, done, info

## 5 · Validate your environment  ⚙️

This cell constructs your env, resets it, takes a few random steps and checks the observation shape and that nothing is NaN — then benchmarks throughput. If a 🧩 method is still unfinished, the traceback here points you straight at it.

In [ ]:
# ✅ Validate YOUR environment before training: correct shapes, finite values, and a
# quick throughput number. If a method still raises NotImplementedError, the traceback
# here tells you which one to finish next.
env = FingerPushVecEnv(XML_PATH, num_envs=2048, device=DEVICE)
obs = env.reset()
assert env.num_obs is not None, "set self.num_obs in FingerPushVecEnv.__init__"
assert obs.shape == (env.num_envs, env.num_obs), (
    f"obs shape {tuple(obs.shape)} != (num_envs={env.num_envs}, num_obs={env.num_obs}) "
    "-- check _compute_obs and self.num_obs")
for _ in range(5):
    a = torch.rand(env.num_envs, env.num_actions, device=DEVICE) * 2 - 1
    obs, rew, done, info = env.step(a)
assert torch.isfinite(obs).all(), "non-finite observation"
assert rew.shape == (env.num_envs,), f"reward shape {tuple(rew.shape)} != (num_envs,)"
print(f"obs {tuple(obs.shape)} | reward sample {rew[:3].tolist()} | checks passed ✅")

# throughput benchmark
for _ in range(5):
    env.step(torch.rand(env.num_envs, env.num_actions, device=DEVICE) * 2 - 1)
torch.cuda.synchronize()
K = 100
t0 = time.time()
for _ in range(K):
    env.step(torch.rand(env.num_envs, env.num_actions, device=DEVICE) * 2 - 1)
torch.cuda.synchronize()
dt = time.time() - t0
print(f"{K} policy steps x {env.num_envs} worlds in {dt:.2f}s "
      f"-> {K*env.num_envs/dt:,.0f} env-steps/s "
      f"({K*env.num_envs*env.decimation/dt:,.0f} physics sub-steps/s)")

## 6&nbsp;· PPO agent

Standard CleanRL-style actor-critic (separate MLPs, state-independent action std).

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias=0.0):
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias)
    return layer

class Agent(nn.Module):
    def __init__(self, n_obs, n_act, hidden=256):
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(n_obs, hidden)), nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)), nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0))
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(n_obs, hidden)), nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)), nn.Tanh(),
            layer_init(nn.Linear(hidden, n_act), std=0.01))
        # start moderately exploratory (std ~= 0.6). The bounds are enforced with
        # .data.clamp_ after each optimizer step, NOT with a clamp() here: clamp's
        # gradient is zero outside the bounds, so a forward-pass clamp freezes the
        # std permanently the moment the parameter drifts past a bound.
        self.actor_logstd = nn.Parameter(torch.full((1, n_act), -0.5))

    def get_value(self, x):
        return self.critic(x).squeeze(-1)

    def get_action_and_value(self, x, action=None):
        mean = self.actor_mean(x)
        std = torch.exp(self.actor_logstd.expand_as(mean))
        dist = torch.distributions.Normal(mean, std)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action).sum(1), dist.entropy().sum(1), self.critic(x).squeeze(-1)


class RunningMeanStd:
    # Vectorised observation normaliser (Welford), kept on the GPU.
    def __init__(self, shape, device):
        self.mean = torch.zeros(shape, device=device)
        self.var = torch.ones(shape, device=device)
        self.count = 1e-4
    def update(self, x):
        b_mean, b_var, b_n = x.mean(0), x.var(0, unbiased=False), x.shape[0]
        delta = b_mean - self.mean
        tot = self.count + b_n
        self.mean += delta * b_n / tot
        m_a = self.var * self.count
        m_b = b_var * b_n
        self.var = (m_a + m_b + delta**2 * self.count * b_n / tot) / tot
        self.count = tot
    def normalize(self, x):
        return (x - self.mean) / torch.sqrt(self.var + 1e-8)


class RewardNormalizer:
    # CleanRL-style NormalizeReward: divide the reward by the running std of the
    # discounted return, then clip. Without this the sparse +50 goal bonus produces
    # return spikes that blow up the value loss (NaNs) once the policy starts scoring.
    def __init__(self, num_envs, gamma, device):
        self.ret = torch.zeros(num_envs, device=device)
        self.rms = RunningMeanStd((), device)
        self.gamma = gamma
    def __call__(self, reward, done):
        self.ret = self.ret * self.gamma + reward
        self.rms.update(self.ret)
        r = reward / torch.sqrt(self.rms.var + 1e-8)
        self.ret = self.ret * (1.0 - done.float())   # reset accumulator on episode end
        return torch.clamp(r, -10.0, 10.0)

## 7 · The PPO trainer  ⚙️

The trainer is provided as a single `train(env, eval_env, …)` function. For this tutorial the important thing about it is what it does **not** contain: there is nothing task-specific inside. **The exact same trainer runs Part A and Part B, unchanged** — every bit of task knowledge lives in the environment *you* wrote.

Because the environment is already vectorised on the GPU, the PPO loop is a plain tensor program: no Gym vector wrappers, no CPU↔GPU copies. Run the **TensorBoard cell** to get a live dashboard (episodic return, reward breakdown, eval distance, throughput), then start training.

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def train(env, eval_env=None, total_steps=16_000_000, run_name="finger_ppo",
          num_steps=24, lr=3e-4, gamma=0.99, gae_lambda=0.95, clip_coef=0.2,
          update_epochs=4, num_minibatches=4, ent_coef=0.0, vf_coef=0.5,
          max_grad_norm=0.5, hidden=256, seed=42, log=True):
    """⚙️ PROVIDED PPO trainer (CleanRL-style, all on the GPU).

    Read it if you're curious, but the point of this tutorial is what it does NOT
    contain: there is NOTHING task-specific in here. The same loop trains Part A and
    Part B unchanged. Everything the policy learns comes from the ENVIRONMENT you
    designed — the observation, the reward, the episode boundaries.

    Because the env is already vectorised on the GPU, the loop is a plain tensor
    program: no Gym vector wrappers, no CPU<->GPU copies. Two subtleties worth knowing:
      * reward + observation normalisation (running mean/std) stabilise learning;
      * on TRUNCATION (time-out) we bootstrap V(final_obs) into the reward, instead of
        treating the time-out as a terminal death — this is why the env separates
        `terminated` from `truncated`.

    Returns (agent, obs_rms, history).
    """
    device = env.device
    torch.manual_seed(seed); np.random.seed(seed)
    b, s, no, na = env.num_envs, num_steps, env.num_obs, env.num_actions
    assert no is not None, "env.num_obs is None — set it in FingerPushVecEnv.__init__"

    agent = Agent(no, na, hidden=hidden).to(device)
    opt = torch.optim.Adam(agent.parameters(), lr=lr, eps=1e-5)
    obs_rms = RunningMeanStd(no, device)
    rew_norm = RewardNormalizer(b, gamma, device)

    mb_obs = torch.zeros(s, b, no, device=device)
    mb_act = torch.zeros(s, b, na, device=device)
    mb_logp = torch.zeros(s, b, device=device)
    mb_rew = torch.zeros(s, b, device=device)
    mb_done = torch.zeros(s, b, device=device)
    mb_val = torch.zeros(s, b, device=device)

    batch_size = b * s
    minibatch_size = batch_size // num_minibatches
    num_iters = total_steps // batch_size

    def norm(o):
        return torch.clamp(obs_rms.normalize(o), -10, 10)

    @torch.no_grad()
    def evaluate():
        # mean closest-approach of the cube to its goal, greedy (mean) policy. Each
        # world is frozen at its first episode end so a mid-eval auto-reset (which
        # would draw a new goal) can't corrupt the number.
        o = eval_env.reset()
        best = torch.full((eval_env.num_envs,), 1e9, device=device)
        active = torch.ones(eval_env.num_envs, dtype=torch.bool, device=device)
        for _ in range(eval_env.steps_per_episode):
            o, _, dn, _ = eval_env.step(agent.actor_mean(norm(o)))
            d = torch.linalg.norm(eval_env.qpos[:, 3:5] - eval_env.target_xy, dim=-1)
            best = torch.where(active, torch.minimum(best, d), best)
            active = active & ~dn
        return best.mean().item()

    writer = SummaryWriter(f"runs/{run_name}") if log else None
    next_obs = env.reset()
    obs_rms.update(next_obs); next_obs_n = norm(next_obs)
    next_done = torch.zeros(b, device=device)
    history = {"step": [], "return": [], "goal_frac": [], "eval_step": [], "eval_cube_dist": []}
    global_step = 0
    t0 = time.time()

    for it in range(1, num_iters + 1):
        opt.param_groups[0]["lr"] = lr * (1.0 - (it - 1) / num_iters)   # linear anneal
        ep_returns, ep_goal = [], []
        term_sums, n_term = {}, 0
        for t in range(num_steps):
            mb_obs[t] = next_obs_n; mb_done[t] = next_done
            with torch.no_grad():
                action, logp, _, value = agent.get_action_and_value(next_obs_n)
            mb_val[t], mb_act[t], mb_logp[t] = value, action, logp

            next_obs, reward, done, info = env.step(action)
            obs_rms.update(next_obs); next_obs_n = norm(next_obs)
            r = rew_norm(reward, done)
            trunc = info["truncated"]
            if trunc.any():
                with torch.no_grad():
                    v_final = agent.get_value(norm(info["final_obs"]))
                r = r + gamma * v_final * trunc.float()   # bootstrap timed-out episodes
            mb_rew[t] = r
            next_done = done.float()
            global_step += b
            if "episode_return" in info:
                ep_returns.append(info["episode_return"]); ep_goal.append(info["goal_reached_frac"])
            for k, v in info["reward_terms"].items():
                term_sums[k] = term_sums.get(k, 0.0) + v
            n_term += 1

        # ---- GAE ----
        with torch.no_grad():
            next_value = agent.get_value(next_obs_n)
            adv = torch.zeros_like(mb_rew); last = torch.zeros(b, device=device)
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    mask, nv = 1.0 - next_done, next_value
                else:
                    mask, nv = 1.0 - mb_done[t + 1], mb_val[t + 1]
                delta = mb_rew[t] + gamma * nv * mask - mb_val[t]
                last = delta + gamma * gae_lambda * mask * last
                adv[t] = last
            returns = adv + mb_val

        # ---- flatten & optimise ----
        f_obs, f_act = mb_obs.reshape(-1, no), mb_act.reshape(-1, na)
        f_logp, f_adv = mb_logp.reshape(-1), adv.reshape(-1)
        f_ret, f_val = returns.reshape(-1), mb_val.reshape(-1)
        idx = np.arange(batch_size)
        for _ in range(update_epochs):
            np.random.shuffle(idx)
            for start in range(0, batch_size, minibatch_size):
                mb = idx[start:start + minibatch_size]
                _, newlogp, entropy, newval = agent.get_action_and_value(f_obs[mb], f_act[mb])
                logratio = torch.clamp(newlogp - f_logp[mb], -10, 10)
                ratio = logratio.exp()
                with torch.no_grad():
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfrac = ((ratio - 1.0).abs() > clip_coef).float().mean()
                a = f_adv[mb]
                a = (a - a.mean()) / (a.std() + 1e-8)
                pg = torch.max(-a * ratio, -a * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)).mean()
                v_loss = 0.5 * ((newval - f_ret[mb]) ** 2).mean()
                loss = pg - ent_coef * entropy.mean() + vf_coef * v_loss
                opt.zero_grad(); loss.backward()
                gn = nn.utils.clip_grad_norm_(agent.parameters(), max_grad_norm)
                if torch.isfinite(loss) and torch.isfinite(gn):   # skip a rare bad minibatch
                    opt.step()
                    agent.actor_logstd.data.clamp_(-4.0, 1.0)      # rails on .data (no blocked grad)

        # ---- logging ----
        sps = int(global_step / (time.time() - t0))
        if writer is not None:
            writer.add_scalar("losses/policy_loss", pg.item(), global_step)
            writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
            writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
            writer.add_scalar("losses/clipfrac", clipfrac.item(), global_step)
            writer.add_scalar("charts/action_std", torch.exp(agent.actor_logstd).mean().item(), global_step)
            writer.add_scalar("charts/learning_rate", opt.param_groups[0]["lr"], global_step)
            writer.add_scalar("charts/SPS", sps, global_step)
            for k in term_sums:
                writer.add_scalar(f"reward/{k}", term_sums[k] / max(n_term, 1), global_step)
        if ep_returns:
            rr, gg = float(np.mean(ep_returns)), float(np.mean(ep_goal))
            history["step"].append(global_step); history["return"].append(rr); history["goal_frac"].append(gg)
            if writer is not None:
                writer.add_scalar("charts/episodic_return", rr, global_step)
                writer.add_scalar("charts/goal_fraction", gg, global_step)

        if eval_env is not None and (it % 10 == 0 or it == num_iters):
            eval_dist = evaluate()
            history["eval_step"].append(global_step); history["eval_cube_dist"].append(eval_dist)
            if writer is not None:
                writer.add_scalar("eval/cube_to_target_dist", eval_dist, global_step)
            rr = history["return"][-1] if history["return"] else float("nan")
            gg = history["goal_frac"][-1] if history["goal_frac"] else float("nan")
            print(f"iter {it:4d}/{num_iters} | step {global_step:>9,} | return {rr:7.3f} | "
                  f"goal {gg:4.2f} | eval cube->target {eval_dist:.3f} m | {sps:,} steps/s", flush=True)

    if writer is not None:
        writer.close()
    print(f"\nTrained {global_step:,} env-steps in {time.time() - t0:.1f}s")
    return agent, obs_rms, history

In [ ]:
# Live dashboard — launch this, then run the training cell; charts update in place.
%load_ext tensorboard
%tensorboard --logdir runs

## 8 · Part A — push to a fixed goal

Every world shares one goal, `(0.30, -0.05)` (the green marker). That spot sits **beyond the finger's static reach** (~0.16 m) while the cube starts ~0.20 m away from it — so the cube has to be *knocked* across and then stopped on the target.

With the **starter reward** the finger reliably learns to drive the cube toward the goal: the eval curve (cube closest-approach) should fall well below the 0.20 m start distance. Getting the cube to arrive slowly and *hold* on the goal is harder — that is exactly what the "improve it" reward-shaping exercise in `_compute_reward` is for. Train first with the starter, watch the result, then come back and add terms.

In [ ]:
# PART A — every world shares the SAME fixed goal (0.30, -0.05).
# 16M steps is a few minutes on an A100/L4, ~10-20 min on a T4. Lower TOTAL for a
# quick look; raise NUM_ENVS until the GPU is full for more diverse data per iteration.
NUM_ENVS, TOTAL = 4096, 16_000_000

env      = FingerPushVecEnv(XML_PATH, num_envs=NUM_ENVS, device=DEVICE, seed=42)
# a small, deterministic held-out set of worlds — a cleaner task metric than the
# (noisy) training return.
eval_env = FingerPushVecEnv(XML_PATH, num_envs=512, device=DEVICE,
                            randomize_init=False, seed=123)

agent, obs_rms, history = train(env, eval_env, total_steps=TOTAL, run_name="finger_ppo_fixed")

### 8b · Learning curves

The cleanest signal is the **eval curve** (left): every 10 iterations we run the *greedy* policy on held-out worlds and measure how close the cube gets to the goal — it should drop well below the 0.20 m start distance. The training return (right, smoothed) is noisier.

In [ ]:
import matplotlib.pyplot as plt


def plot_curves(history, title="Part A"):
    def smooth(x, k=9):
        x = np.asarray(x, dtype=float)
        return np.convolve(x, np.ones(k) / k, mode="valid") if len(x) >= k else x

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history["eval_step"], history["eval_cube_dist"], marker="o", ms=3, color="tab:red")
    ax[0].axhline(0.20, ls="--", c="gray", lw=1, label="start distance")
    ax[0].axhline(0.05, ls=":", c="gray", lw=1, label="goal radius")
    ax[0].set_title(f"{title}: eval cube closest-approach to goal")
    ax[0].set_ylabel("metres"); ax[0].set_xlabel("env steps"); ax[0].legend(); ax[0].grid(alpha=.3)

    if len(history["return"]) > 9:
        s = smooth(history["return"])
        ax[1].plot(history["step"][len(history["step"]) - len(s):], s, color="tab:blue")
    else:
        ax[1].plot(history["step"], history["return"], color="tab:blue")
    ax[1].set_title("Training episodic return (smoothed)")
    ax[1].set_xlabel("env steps"); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()


plot_curves(history, title="Part A")

### 8c · Watch it, and evaluate

Warp is built for throughput, not rendering, so we roll the policy out in a single **CPU** MuJoCo copy (identical model, observation and PD controller) and render off-screen. Then we report clean task metrics on the held-out worlds and save a checkpoint.

In [ ]:
import mediapy as media


def rollout_video(agent, obs_rms, goals, steps_per_goal=200, camera=None):
    """⚙️ PROVIDED. Roll out the GREEDY policy in a single CPU MuJoCo copy (identical
    model + PD controller) and render it off-screen — Warp is built for throughput, not
    pretty pictures. Plays one segment per goal in `goals` (a list of (x, y) targets),
    moving the green marker to each so you can watch the finger push the red cube there.

    NOTE: this rebuilds the observation BY HAND, in the same order as
    FingerPushVecEnv._compute_obs. If you changed that order, mirror the change in
    get_obs() below.
    """
    mjm = mujoco.MjModel.from_xml_path(XML_PATH)
    mjd = mujoco.MjData(mjm)
    renderer = mujoco.Renderer(mjm, height=480, width=640)
    tip_bid = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "finger_lower_link")
    dj = np.array([0.0, 0.5, -0.75])
    kp, kd, ascale, deci, tlim = 2.5, 0.2, 0.5, 4, 1.0

    def norm(o):
        return torch.clamp(obs_rms.normalize(o), -10, 10)

    def get_obs(target_xy, prev_action):
        jp, jv = mjd.qpos[:3], mjd.qvel[:3]
        cp, cq = mjd.qpos[3:6], mjd.qpos[6:10]
        clv, cav = mjd.qvel[3:6], mjd.qvel[6:9]
        cube_pos_rel = cp[:2] - target_xy               # <-- same design decision as _compute_obs
        tip_to_cube = cp - mjd.xpos[tip_bid]
        return np.concatenate([jp, jv, cube_pos_rel, cq, clv, cav,
                               tip_to_cube, prev_action]).astype(np.float32)

    frames = []
    for gx, gy in goals:
        target_xy = np.array([gx, gy])
        mujoco.mj_resetData(mjm, mjd)
        mjm.geom("target_marker").pos[:] = [gx, gy, 0.05]   # move the visual goal marker
        prev_action = np.zeros(3)
        mujoco.mj_forward(mjm, mjd)
        for _ in range(steps_per_goal):
            o = torch.tensor(get_obs(target_xy, prev_action), device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                mean = agent.actor_mean(norm(o))[0].cpu().numpy()   # deterministic action
            mean = np.clip(mean, -1.0, 1.0)
            target = ascale * mean + dj
            for _ in range(deci):
                mjd.ctrl[:3] = np.clip(kp * (target - mjd.qpos[:3]) - kd * mjd.qvel[:3], -tlim, tlim)
                mujoco.mj_step(mjm, mjd)
            prev_action = mean
            renderer.update_scene(mjd, camera=camera)
            frames.append(renderer.render())
    renderer.close()
    return frames

In [ ]:
# Watch the Part A policy push the cube to the single fixed goal.
frames = rollout_video(agent, obs_rms, goals=[(0.30, -0.05)], steps_per_goal=250)
media.write_video("rollout_partA.gif", frames[::2], fps=0.5 / env.control_dt, codec="gif")
media.show_video(frames, fps=1.0 / env.control_dt)

In [ ]:
# Greedy evaluation on the held-out worlds + save a checkpoint (weights are useless
# without the obs-normaliser stats, so we save both — same format as
# scripts/play_finger_warp.py).
import json, os


def norm(o):
    return torch.clamp(obs_rms.normalize(o), -10, 10)


@torch.no_grad()
def report(eval_env, agent, obs_rms):
    o = eval_env.reset()
    best = torch.full((eval_env.num_envs,), 1e9, device=DEVICE)
    solved = torch.zeros(eval_env.num_envs, dtype=torch.bool, device=DEVICE)
    active = torch.ones(eval_env.num_envs, dtype=torch.bool, device=DEVICE)
    for _ in range(eval_env.steps_per_episode - 1):
        o, _, dn, inf = eval_env.step(agent.actor_mean(norm(o)))
        d = torch.linalg.norm(eval_env.qpos[:, 3:5] - eval_env.target_xy, dim=-1)
        best = torch.where(active, torch.minimum(best, d), best)
        solved |= inf["goal_reached"] & active
        active = active & ~dn
    print(f"eval over {eval_env.num_envs} worlds (start distance ~0.20 m):")
    print(f"  closest approach   mean {best.mean():.3f} m | median {best.median():.3f} m")
    print(f"  within 5 cm goal   {100 * (best < eval_env.goal_radius).float().mean():.1f}% of worlds")
    print(f"  held on the goal   {100 * solved.float().mean():.1f}% of worlds")


report(eval_env, agent, obs_rms)

os.makedirs("runs/finger_ppo_fixed", exist_ok=True)
torch.save({"agent": {k: v.detach().cpu() for k, v in agent.state_dict().items()},
            "obs_mean": obs_rms.mean.cpu(), "obs_var": obs_rms.var.cpu(),
            "num_obs": env.num_obs, "num_actions": env.num_actions},
           "runs/finger_ppo_fixed/policy.pt")
with open("runs/finger_ppo_fixed/history.json", "w") as f:
    json.dump(history, f)
print("saved checkpoint -> runs/finger_ppo_fixed/policy.pt")

## 9 · Part B — push to *any* goal (goal-conditioned)

A policy that only ever saw one goal has simply memorised one motion. Now we want **one** policy that pushes the cube to a goal **sampled fresh every episode**. This is a *goal-conditioned* policy, and it surfaces the single most important idea in this tutorial:

> **If the goal changes between episodes, the goal must be part of the observation.**
> Otherwise two episodes with the same cube state but different goals require different actions from the *same* observation — the task is partially observed, and no policy can solve it.

Here is the elegant part: **you already did this.** In `_compute_obs` you encoded the cube position *relative to the goal* (`cube_pos_rel = cube_pos − target_xy`). Move the goal and that number changes, so the network can already tell where it is being asked to push. The observation, reward, controller and done-logic **do not need to change at all.**

So the **entire** change for Part B is *how goals are drawn*: instead of one fixed goal, sample a random one per world on reset. We express that by **subclassing** and overriding a single method, `_resample_goals`. Two things the provided code already handles for you:

- **Per-world start-distance.** The distance reward is normalised by `d_init` (the cube→goal distance at reset). Because each world now has its own goal, `_init_world_state` recomputes `d_init` per world automatically — no change needed.
- **The visual marker** is moved to each goal inside the rollout helper.

That "goal-conditioned almost for free" outcome is your reward for designing the observation well in Part A. Encode the goal *absolutely* instead, and Part B would have forced you to rewrite the observation and retune everything.

### 🧩 Your task — sample goals

Complete `_resample_goals` in the subclass below: for each resetting world, draw a random goal from a rectangular region (`goal_x_range` × `goal_y_range`) and write it into `self.target_xy[ids]`. That is the *only* new code in Part B.

In [ ]:
class GoalConditionedFingerPushVecEnv(FingerPushVecEnv):
    """Part B: every world (re)draws a RANDOM goal on reset, so the policy must learn
    to push the cube to WHEREVER the goal is — not to one memorised spot.

    Look how little changes. We override exactly ONE method, _resample_goals. We do
    NOT touch the observation, reward, controller or done logic. That works because:
      * _compute_obs already encodes the cube RELATIVE TO THE GOAL, so the network is
        automatically goal-aware — a different goal simply produces a different obs;
      * the provided _init_world_state recomputes the per-world start-distance d_init
        for the new goal automatically.
    This is the payoff of designing the environment well in Part A.
    """

    def __init__(self, *args, goal_x_range=(0.24, 0.34), goal_y_range=(-0.14, 0.04),
                 **kwargs):
        # store the ranges BEFORE super().__init__, because it calls _resample_goals
        self.goal_x_range = goal_x_range
        self.goal_y_range = goal_y_range
        super().__init__(*args, **kwargs)

    def _resample_goals(self, ids):
        """🧩 YOUR CODE (Part B): draw a fresh random goal for each world in `ids` and
        write it into self.target_xy[ids] (shape (len(ids), 2)). Sample x uniformly in
        self.goal_x_range and y uniformly in self.goal_y_range. Use self.gen so the
        draws are reproducible:

            n  = ids.shape[0]
            u  = torch.rand(n, 2, generator=self.gen, device=self.device)   # in [0,1)
            xlo, xhi = self.goal_x_range
            ylo, yhi = self.goal_y_range
            self.target_xy[ids, 0] = xlo + (xhi - xlo) * u[:, 0]
            self.target_xy[ids, 1] = ylo + (yhi - ylo) * u[:, 1]
        """
        raise NotImplementedError("implement _resample_goals for Part B (then delete this line)")

### Train the goal-conditioned policy

Same `train(...)`, same hyperparameters as Part A — only the environment changed. A few notes on the goal region below:

- **Reachability.** The finger base is fixed at the origin and pushes the cube *outward* (+x). Goals biased toward +x with moderate y are pushable; goals behind the cube or far to the side are much harder. The default region is a sensible starting box — **if the success rate stays low, shrink it** (a simple curriculum) or bias it toward +x.
- Goal-conditioned tasks are a little harder than a single goal, so this trains for more steps. Lower `TOTAL` for a quick look.

In [ ]:
# PART B — each world gets a RANDOM goal, resampled every reset. Same trainer, same
# hyperparameters as Part A: only the environment changed. Goal-conditioned tasks are
# a bit harder, so give it more steps.
NUM_ENVS, TOTAL = 4096, 32_000_000
GOAL_X, GOAL_Y = (0.24, 0.34), (-0.14, 0.04)   # the region goals are drawn from (tune me)

gc_env      = GoalConditionedFingerPushVecEnv(XML_PATH, num_envs=NUM_ENVS, device=DEVICE,
                                              goal_x_range=GOAL_X, goal_y_range=GOAL_Y, seed=42)
gc_eval_env = GoalConditionedFingerPushVecEnv(XML_PATH, num_envs=512, device=DEVICE,
                                              goal_x_range=GOAL_X, goal_y_range=GOAL_Y,
                                              randomize_init=False, seed=123)

agent_gc, obs_rms_gc, history_gc = train(gc_env, gc_eval_env, total_steps=TOTAL,
                                         run_name="finger_ppo_goalcond")
plot_curves(history_gc, title="Part B")

### Generalisation across goals

Does *one* policy really solve *many* goals? We evaluate over a fresh batch of random goals and plot each goal location coloured by how close the cube got — a map of where the policy is strong and where it struggles (often near the edges of the training region).

In [ ]:
# Does one policy really solve MANY goals? Evaluate over a fresh batch of random goals
# and plot each goal location coloured by how close the cube got — a map of where the
# policy generalises and where it struggles.
import matplotlib.pyplot as plt


def norm_gc(o):
    return torch.clamp(obs_rms_gc.normalize(o), -10, 10)

with torch.no_grad():
    o = gc_eval_env.reset()
    goals = gc_eval_env.target_xy.clone()                # goal per world (frozen at reset)
    best = torch.full((gc_eval_env.num_envs,), 1e9, device=DEVICE)
    solved = torch.zeros(gc_eval_env.num_envs, dtype=torch.bool, device=DEVICE)
    active = torch.ones(gc_eval_env.num_envs, dtype=torch.bool, device=DEVICE)
    for _ in range(gc_eval_env.steps_per_episode - 1):
        o, _, dn, inf = gc_eval_env.step(agent_gc.actor_mean(norm_gc(o)))
        d = torch.linalg.norm(gc_eval_env.qpos[:, 3:5] - gc_eval_env.target_xy, dim=-1)
        best = torch.where(active, torch.minimum(best, d), best)
        solved |= inf["goal_reached"] & active
        active = active & ~dn

print(f"goal-conditioned eval over {gc_eval_env.num_envs} random goals:")
print(f"  mean closest approach {best.mean():.3f} m | within 5 cm {100*(best<0.05).float().mean():.1f}%")
print(f"  held on the goal      {100*solved.float().mean():.1f}% of worlds")

g, bb = goals.cpu().numpy(), best.cpu().numpy()
plt.figure(figsize=(5.2, 5))
sc = plt.scatter(g[:, 0], g[:, 1], c=np.minimum(bb, 0.2), cmap="viridis_r", s=14)
plt.colorbar(sc, label="closest approach (m)")
plt.scatter([gc_eval_env.cube_init_xy[0].item()], [gc_eval_env.cube_init_xy[1].item()],
            marker="s", c="red", s=60, edgecolor="k", label="cube start", zorder=3)
plt.gca().set_aspect("equal"); plt.xlabel("goal x (m)"); plt.ylabel("goal y (m)")
plt.title("Part B: generalisation across the goal region"); plt.legend()
plt.tight_layout(); plt.show()

### Watch one policy solve several goals

The same network, pushing the cube to four different targets in a row — the green marker moves each time.

In [ ]:
# Watch the SAME policy push the cube to several different goals in a row.
demo_goals = [(0.26, -0.10), (0.32, 0.02), (0.30, -0.13), (0.24, 0.00)]
frames = rollout_video(agent_gc, obs_rms_gc, goals=demo_goals, steps_per_goal=150)
media.write_video("rollout_partB.gif", frames[::2], fps=0.5 / gc_env.control_dt, codec="gif")
media.show_video(frames, fps=1.0 / gc_env.control_dt)

## 10 · Where to go from here

**Improve the reward (Part A).** The starter reaches toward the goal but tends to overshoot and never settle. Add the terms hinted in `_compute_reward` — *precision* (a sharp gradient in the last few cm), *brake* (punish cube speed near the goal so it parks instead of smacking through), a sparse *hold bonus* (the real success signal), *action-rate* smoothing — and watch each one separately in TensorBoard. Getting the cube to **stop and hold** on the goal is the genuinely hard part of this task.

**Harder goals (Part B).**
- **Curriculum.** Start with a large `goal_radius` (e.g. 0.10) and/or a small goal region so the sparse hold-bonus actually fires, then shrink both as the success rate climbs.
- **Full pose.** Sample a target *orientation* too and reward reaching it — nonprehensile reorientation by pushing, a much harder control problem and a great extension.
- **Domain randomisation.** With thousands of parallel worlds it is essentially free to randomise object mass / friction / size / shape per world for a more robust policy (`target_xy` already shows the per-world pattern).

**Scale.** Increase `NUM_ENVS` until the GPU is full — more parallel worlds means more diverse data per iteration and faster wall-clock convergence.